In [1]:
# Importing torch
import torch
import torchvision

print(torch.__version__)
print(torchvision.__version__)

2.11.0+cu126
0.26.0+cu126


In [2]:
# GPU avaliability
if torch.cuda.is_available():
    print("GPU is available!")
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("GPU not available. Using CPU.")

GPU is available!
Using GPU: NVIDIA GeForce MX350


#### Tensor Creation Methods

In [3]:
# This method will assign a memory to create a tensor of provided shape and return the existing numbers at that location
a = torch.empty(size=[2, 3], dtype=torch.float)
a

tensor([[0., 0., 0.],
        [0., 0., 0.]])

Scaler (1D) → Vector (2D) → Tensor (> 2D)

In [4]:
# Check type
type(a)

torch.Tensor

In [5]:
# Check the memory consuption
from sys import getsizeof

print(f"Total Memory {getsizeof(a)}")
print(f"Tensor Memory {a.nbytes}") # Float of 4 bytes

Total Memory 72
Tensor Memory 24


*The core difference is that sys.getsizeof() measures the total memory footprint of an object (including overhead and references), while .nbytes measures only the raw data size of the object's elements, excluding overhead.*

In [6]:
# Using zeros
torch.zeros(size = [2, 3], dtype = torch.float32)

tensor([[0., 0., 0.],
        [0., 0., 0.]])

In [7]:
# using ones
torch.ones(size = [2, 3])

tensor([[1., 1., 1.],
        [1., 1., 1.]])

In [8]:
# using rand
torch.rand(size = [2, 3])

tensor([[0.6995, 0.4340, 0.4126],
        [0.7380, 0.2369, 0.8468]])

In [9]:
# manual_seed - for regenerating the same array
seed_number = torch.initial_seed() # Store this seed number and use it as a variable
torch.manual_seed(seed_number)
torch.rand(2,3)

tensor([[0.6995, 0.4340, 0.4126],
        [0.7380, 0.2369, 0.8468]])

In [10]:
# Device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [11]:
# using tensor

torch.tensor(
    data = [[1,2,3],[4,5,6]],
    dtype = torch.float32,
    requires_grad = False,
    device = device,
    pin_memory = True # The tensor will directly loded into pin-memory and not pager-memory, for faster loading. pin_memory=True is primarily a GPU optimization tool that prepares CPU data for faster transfer to the GPU.
)

tensor([[1., 2., 3.],
        [4., 5., 6.]], device='cuda:0')

*When you create a regular tensor, your operating system can move its data between RAM and disk (virtual memory). This is efficient for general use but unpredictable for transfers to a GPU.*

*Setting pin_memory=True tells PyTorch to allocate the tensor in page-locked (pinned) memory. This guarantees the data stays in RAM, enabling the GPU's Direct Memory Access (DMA) to perform a direct and fast transfer. Without it, the system must create a temporary pinned copy first, which is slower.*

*Pinned memory (page-locked memory) is a feature of CPU memory that allows for faster, asynchronous data transfers to the GPU.*
*You can only pin tensors that are currently on the CPU.*
*Once a tensor is already on the GPU (device='cuda'), it cannot be "pinned" because it is no longer in system RAM.*

[A guide on good usage of non_blocking and pin_memory() in PyTorch](https://docs.pytorch.org/tutorials/intermediate/pinmem_nonblock.html)

In [12]:
# arange
print("using arange ->", torch.arange(start=0, end=10, step=2))

# using linspace
print("using linspace ->", torch.linspace(start=0, end=10, steps=10))

# using eye - Identity Matrix tensor
print("using eye ->", torch.eye(n=5))

# using full - torch.ones(size = (m, n)) * constant( = 5)
print("using full ->", torch.full(size=(3, 3), fill_value=5))

using arange -> tensor([0, 2, 4, 6, 8])
using linspace -> tensor([ 0.0000,  1.1111,  2.2222,  3.3333,  4.4444,  5.5556,  6.6667,  7.7778,
         8.8889, 10.0000])
using eye -> tensor([[1., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0.],
        [0., 0., 1., 0., 0.],
        [0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 1.]])
using full -> tensor([[5, 5, 5],
        [5, 5, 5],
        [5, 5, 5]])


---

#### Tensor Shape

In [13]:
x = torch.tensor([[1,2,3],[4,5,6]])
x

tensor([[1, 2, 3],
        [4, 5, 6]])

In [14]:
print(x.shape)
print(x.size(dim=1)) # You can pass the dimention as well → dim = 0 (rows - outermost dimention) 

torch.Size([2, 3])
3


In [15]:
# Create an uninitialized tensor with the same shape as the input tensor
y = torch.empty_like(
    input=x,
    memory_format=torch.preserve_format, # The individual elements in this tensor pointing to the same location where the elements of first tensor is pointing -> else some random numbers with equal shape 
    dtype=torch.float32,
    layout=x.layout, # optional
    device=device,
    pin_memory=False, # Since you are creating a tensor directly on GPU, it will get stored in GPU memory and not RAM. Which means there is no transfer from RAM to GPU memory → Only pin if it's on CPU
    requires_grad=True
)

print(y)

tensor([[1., 2., 3.],
        [4., 5., 6.]], device='cuda:0', requires_grad=True)


In [16]:
# torch.preserve_format
print(id(x[0][0]))
print(id(y[0][0]))

2286740178416
2286740180576


In [17]:
torch.zeros_like(x)

tensor([[0, 0, 0],
        [0, 0, 0]])

In [18]:
torch.ones_like(x)

tensor([[1, 1, 1],
        [1, 1, 1]])

In [19]:
torch.rand_like(x, dtype=torch.float32)

tensor([[0.6157, 0.7993, 0.1615],
        [0.7008, 0.8641, 0.5206]])

*`_like()` These methods create a new tensor with the exact same shape, dtype, and device as an existing input tensor.*

---

#### Tensor DataTypes

In [20]:
# find data type
x.dtype

torch.int64

In [21]:
# assign data type
torch.tensor([1.0, 2.0, 3.0], dtype = torch.int32)

tensor([1, 2, 3], dtype=torch.int32)

In [22]:
torch.tensor([1, 2, 3], dtype = torch.float64)

tensor([1., 2., 3.], dtype=torch.float64)

In [23]:
# using to() - Type Conversion - Generally used to transfer a tensor from cpu to gpu
x.to(torch.float32)

tensor([[1., 2., 3.],
        [4., 5., 6.]])

| **Data Type**             | **Dtype**         | **Description**                                                                                                                                                                |
|---------------------------|-------------------|--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| **32-bit Floating Point** | `torch.float32`   | Standard floating-point type used for most deep learning tasks. Provides a balance between precision and memory usage.                                                         |
| **64-bit Floating Point** | `torch.float64`   | Double-precision floating point. Useful for high-precision numerical tasks but uses more memory.                                                                               |
| **16-bit Floating Point** | `torch.float16`   | Half-precision floating point. Commonly used in mixed-precision training to reduce memory and computational overhead on modern GPUs.                                            |
| **BFloat16**              | `torch.bfloat16`  | Brain floating-point format with reduced precision compared to `float16`. Used in mixed-precision training, especially on TPUs.                                                |
| **8-bit Floating Point**  | `torch.float8`    | Ultra-low-precision floating point. Used for experimental applications and extreme memory-constrained environments (less common).                                               |
| **8-bit Integer**         | `torch.int8`      | 8-bit signed integer. Used for quantized models to save memory and computation in inference.                                                                                   |
| **16-bit Integer**        | `torch.int16`     | 16-bit signed integer. Useful for special numerical tasks requiring intermediate precision.                                                                                    |
| **32-bit Integer**        | `torch.int32`     | Standard signed integer type. Commonly used for indexing and general-purpose numerical tasks.                                                                                  |
| **64-bit Integer**        | `torch.int64`     | Long integer type. Often used for large indexing arrays or for tasks involving large numbers.                                                                                  |
| **8-bit Unsigned Integer**| `torch.uint8`     | 8-bit unsigned integer. Commonly used for image data (e.g., pixel values between 0 and 255).                                                                                    |
| **Boolean**               | `torch.bool`      | Boolean type, stores `True` or `False` values. Often used for masks in logical operations.                                                                                      |
| **Complex 64**            | `torch.complex64` | Complex number type with 32-bit real and 32-bit imaginary parts. Used for scientific and signal processing tasks.                                                               |
| **Complex 128**           | `torch.complex128`| Complex number type with 64-bit real and 64-bit imaginary parts. Offers higher precision but uses more memory.                                                                 |
| **Quantized Integer**     | `torch.qint8`     | Quantized signed 8-bit integer. Used in quantized models for efficient inference.                                                                                              |
| **Quantized Unsigned Integer** | `torch.quint8` | Quantized unsigned 8-bit integer. Often used for quantized tensors in image-related tasks.                                                                                     |


---

#### Mathematical Operations

In [24]:
x = torch.rand(2,2)
x

tensor([[0.1698, 0.9941],
        [0.3387, 0.8895]])

In [25]:
# Scalar Operations

# addition
print(x + 2, end = "\n\n")

# substraction
print(x - 2, end = "\n\n")

# multiplication
print(x * 3, end = "\n\n")

# division
print(x / 3, end = "\n\n")

# int division
print((x * 100) // 3, end = "\n\n")

# mod
print(((x * 100) // 3) % 2, end = "\n\n")

# power
print(x ** 2)

tensor([[2.1698, 2.9941],
        [2.3387, 2.8895]])

tensor([[-1.8302, -1.0059],
        [-1.6613, -1.1105]])

tensor([[0.5093, 2.9822],
        [1.0161, 2.6684]])

tensor([[0.0566, 0.3314],
        [0.1129, 0.2965]])

tensor([[ 5., 33.],
        [11., 29.]])

tensor([[1., 1.],
        [1., 1.]])

tensor([[0.0288, 0.9881],
        [0.1147, 0.7912]])


In [26]:
# floor division
(torch.tensor([1.0, 2.0, 3.0]) * 100) // 3

tensor([ 33.,  66., 100.])

In [27]:
# Element wise operation
a = torch.rand(2,3)
b = torch.rand(2,3)

print(a)
print(b)

tensor([[0.0630, 0.6349, 0.5378],
        [0.8630, 0.2240, 0.5266]])
tensor([[0.8156, 0.8821, 0.0307],
        [0.3872, 0.4336, 0.2974]])


In [28]:
# Shape must be same or able to broadcast

# add
print(a + b, end="\n\n")

# sub
print(a - b, end="\n\n")

# multiply
print(a * b, end="\n\n")

# division
print(a / b, end="\n\n")

# power
print(a ** b, end="\n\n")

# mod
print(a % b)

tensor([[0.8786, 1.5170, 0.5685],
        [1.2502, 0.6576, 0.8240]])

tensor([[-0.7526, -0.2472,  0.5072],
        [ 0.4759, -0.2096,  0.2293]])

tensor([[0.0514, 0.5600, 0.0165],
        [0.3341, 0.0971, 0.1566]])

tensor([[ 0.0772,  0.7197, 17.5367],
        [ 2.2291,  0.5165,  1.7710]])

tensor([[0.1049, 0.6698, 0.9812],
        [0.9446, 0.5227, 0.8264]])

tensor([[0.0630, 0.6349, 0.0165],
        [0.0887, 0.2240, 0.2293]])


In [29]:
c = torch.tensor([1, -2, 3, -4])
c

tensor([ 1, -2,  3, -4])

In [30]:
# abs
torch.abs(c)

tensor([1, 2, 3, 4])

In [31]:
# negative
torch.neg(c)

tensor([-1,  2, -3,  4])

In [32]:
# Inplace operations
print(torch.abs_(c))
print(c)

print(torch.neg_(c))
print(c)

tensor([1, 2, 3, 4])
tensor([1, 2, 3, 4])
tensor([-1, -2, -3, -4])
tensor([-1, -2, -3, -4])


In [33]:
c.fill_(value=10)
# In-place equivalent → c = torch.full_like(c, fill_value=10)

tensor([10, 10, 10, 10])

In [63]:
x = torch.empty_like(input=c)
c.copy_(other=x) # c gets updated, In-place equivalent → c = x.clone()
print(c)

tensor([      3565024249632, 4446636817168914816, 4497618955177123024,
                          0])


In [38]:
d = torch.tensor([1.9, 2.3, 3.7, 4.4, 6.5])
d

tensor([1.9000, 2.3000, 3.7000, 4.4000, 6.5000])

In [39]:
# round
torch.round(d) # 6.5 -> 6 (get floored not ceiled)

tensor([2., 2., 4., 4., 6.])

In [40]:
# ceil - nearest bigger integer value
torch.ceil(d)

tensor([2., 3., 4., 5., 7.])

In [41]:
# floor - nearest smaller integer value
torch.floor(d)

tensor([1., 2., 3., 4., 6.])

In [42]:
# clamp | clip - Clipping the values in a certing range
torch.clip(d, min = -1, max = 1)
torch.clamp(d, min = -1, max = 1)

tensor([1., 1., 1., 1., 1.])

In [43]:
e = torch.randint(size=(2,3), low=0, high=10, dtype=torch.float32)
e

tensor([[5., 2., 1.],
        [5., 2., 9.]])

In [44]:
# Aggregate operation

# sum
print(torch.sum(e))

# sum along columns
print(torch.sum(e, dim = 0, keepdim=False)) # same as `axis` parameter in numpy

# sum along rows
print(torch.sum(e, dim = 1, keepdim=False))

# keepdim = True
print(torch.sum(e, dim = 1, keepdim=True))

tensor(24.)
tensor([10.,  4., 10.])
tensor([ 8., 16.])
tensor([[ 8.],
        [16.]])


*If `keepdim` is `False`, the output tensor will not retain the dimension across which the sum was performed.*

In [45]:
# mean
torch.mean(e)

# mean along col
torch.mean(e, dim=0)

tensor([5., 2., 5.])

In [46]:
# median
torch.median(e)

tensor(2.)

In [47]:
# max and min
torch.max(e)
torch.min(e)

tensor(1.)

In [48]:
# product
torch.prod(e) # .mul() is a element-wise multiplication between two matrics

tensor(900.)

In [49]:
# standard deviation
torch.std(e)

tensor(2.9665)

In [50]:
# variance
torch.var(e)

tensor(8.8000)

In [51]:
# argmax
torch.argmax(e)

tensor(5)

In [52]:
# argmin
torch.argmin(e)

tensor(2)

In [67]:
# More _methods()
a = torch.full_like(input=e, fill_value=10)
b = a.clone().copy_(a) * 100
print(a, b, sep="\n")

a.clone().add_(b) # a = a + b (ignore clone)
a.clone().sub_(b) # a = a - b 
a.clone().mul_(b) # a = a * b (element wise)

a.zero_() # x = torch.zeros_like(x)

tensor([[10., 10., 10.],
        [10., 10., 10.]])
tensor([[1000., 1000., 1000.],
        [1000., 1000., 1000.]])


tensor([[0., 0., 0.],
        [0., 0., 0.]])

---

#### Matrix operations

In [68]:
f = torch.randint(size=(2,3), low=0, high=10)
g = torch.randint(size=(3,2), low=0, high=10)

print(f)
print(g)

tensor([[0, 9, 3],
        [3, 7, 7]])
tensor([[0, 1],
        [0, 9],
        [1, 9]])


In [69]:
# matrix multiplcation - dot product
torch.matmul(f, g)

tensor([[  3, 108],
        [  7, 129]])

In [70]:
vector1 = torch.tensor([1, 2])
vector2 = torch.tensor([3, 4])

# dot product - only works with 1D Tensors
torch.dot(vector1, vector2)

tensor(11)

In [71]:
# transpose
torch.transpose(f, 0, 1)

tensor([[0, 3],
        [9, 7],
        [3, 7]])

In [ ]:
torch.transpose(f, dim0 = 1, dim1 = 0) # Transpose is same as `reshape` operation by swapping the dimentions

tensor([[3, 6],
        [4, 2],
        [9, 0]])

In [74]:
# Implace Transpose
f.t_()

tensor([[0, 3],
        [9, 7],
        [3, 7]])

In [75]:
temp = torch.tensor([
    [[1], [2]],
    [[1], [3]],
    [[1], [4]],
])

print(temp.shape)
torch.transpose(temp, dim0=0, dim1=1).shape # Check the dimentions and its index correctly

torch.Size([3, 2, 1])


torch.Size([2, 3, 1])

*Tip: To visualize a transpose, flatten the tensor into a 1D array and then re-arrange the elements according to the target shape.*

*For a tensor with shape [2, 3, 1], the 0th dimension has size 2, the 1st dimension has size 3, and so on.*

In [77]:
# Convert any nD tensor to 1D tensor
print(torch.tensor([[1, 2], [3, 4]]).ravel())
print(torch.tensor([[1, 2], [3, 4]]).flatten())
print(torch.tensor([[[1], [2]], [[3], [4]]]).reshape(-1))

tensor([1, 2, 3, 4])
tensor([1, 2, 3, 4])
tensor([1, 2, 3, 4])


In [78]:
h = torch.randint(size=(3,3), low=0, high=10, dtype=torch.float32)
h

tensor([[3., 8., 6.],
        [9., 8., 1.],
        [9., 1., 0.]])

In [79]:
# determinant
torch.det(h)

tensor(-309.)

In [80]:
# inverse
torch.inverse(h)

tensor([[ 0.0032, -0.0194,  0.1294],
        [-0.0291,  0.1748, -0.1650],
        [ 0.2039, -0.2233,  0.1553]])

---

#### Comparison operations

In [81]:
i = torch.randint(size=(2,3), low=0, high=10)
j = torch.randint(size=(2,3), low=0, high=10)

print(i)
print(j)

tensor([[4, 9, 9],
        [2, 9, 9]])
tensor([[2, 8, 1],
        [9, 5, 4]])


In [82]:
# Element wise comparison - Shape must be same or able to broadcast

# greater than
print(i > j)

# less than
print(i < j)

# equal to
print(i == j)

# not equal to
print(i != j)

tensor([[ True,  True,  True],
        [False,  True,  True]])
tensor([[False, False, False],
        [ True, False, False]])
tensor([[False, False, False],
        [False, False, False]])
tensor([[True, True, True],
        [True, True, True]])


In [84]:
# Boolean Indexing - Reshaped into 1D
i[i > j]

tensor([4, 9, 9, 9, 9])

---

#### Special functions

In [85]:
k = torch.randint(size=(2,3), low=0, high=10, dtype=torch.float64)
k

tensor([[5., 6., 2.],
        [4., 7., 5.]], dtype=torch.float64)

In [86]:
# log - base e
torch.log(input = k)

# log - base 10
torch.log10(input = k)

tensor([[0.6990, 0.7782, 0.3010],
        [0.6021, 0.8451, 0.6990]], dtype=torch.float64)

In [ ]:
# Checking the size of the elements in the tensor
tensor = torch.log10(input = k)
tensor.element_size()  # size in bytes of a single element

8

In [91]:
# exp
torch.exp(k)

tensor([[ 148.4132,  403.4288,    7.3891],
        [  54.5982, 1096.6332,  148.4132]], dtype=torch.float64)

In [92]:
# sqrt
torch.sqrt(k)

tensor([[2.2361, 2.4495, 1.4142],
        [2.0000, 2.6458, 2.2361]], dtype=torch.float64)

In [93]:
# sigmoid
torch.sigmoid(k)

tensor([[0.9933, 0.9975, 0.8808],
        [0.9820, 0.9991, 0.9933]], dtype=torch.float64)

In [94]:
# softmax
torch.softmax(k, dim = 1)

tensor([[0.2654, 0.7214, 0.0132],
        [0.0420, 0.8438, 0.1142]], dtype=torch.float64)

In [95]:
# relu
torch.relu(k)

tensor([[5., 6., 2.],
        [4., 7., 5.]], dtype=torch.float64)

---

#### Inplace Operations

In [96]:
m = torch.rand(2,3)
n = torch.rand(2,3)

print(m)
print(n)

tensor([[0.2801, 0.7260, 0.5001],
        [0.0736, 0.3499, 0.5936]])
tensor([[0.3870, 0.4457, 0.6023],
        [0.6001, 0.1346, 0.8339]])


In [97]:
# _ methods
m.add_(n) # It returns the new matrix

tensor([[0.6671, 1.1718, 1.1025],
        [0.6738, 0.4845, 1.4275]])

In [98]:
m # Updated

tensor([[0.6671, 1.1718, 1.1025],
        [0.6738, 0.4845, 1.4275]])

In [99]:
n

tensor([[0.3870, 0.4457, 0.6023],
        [0.6001, 0.1346, 0.8339]])

In [100]:
torch.relu(m)

tensor([[0.6671, 1.1718, 1.1025],
        [0.6738, 0.4845, 1.4275]])

In [101]:
m.relu_()

tensor([[0.6671, 1.1718, 1.1025],
        [0.6738, 0.4845, 1.4275]])

In [102]:
m

tensor([[0.6671, 1.1718, 1.1025],
        [0.6738, 0.4845, 1.4275]])

---

#### Copying a Tensor

In [103]:
a = torch.tensor(
    data = [[1,2,3], [4,5,6]]
)
b = a.clone()

print(a, b, sep = '\n')
print(id(a), id(b)) # Not same

tensor([[1, 2, 3],
        [4, 5, 6]])
tensor([[1, 2, 3],
        [4, 5, 6]])
2286955111968 2286955109648


In [104]:
a[0][0] = 10

print(a, b, sep = '\n')

tensor([[10,  2,  3],
        [ 4,  5,  6]])
tensor([[1, 2, 3],
        [4, 5, 6]])


In [105]:
id(a)

2286955111968

In [106]:
id(b)

2286955109648

---

A third-order polynomial, trained to predict `y = sin(x) from -pi to pi` by minimizing squared Euclidean distance.

In [ ]:
# Variables Initialization
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float
learning_rate = 1e-6

# Dataset
x = torch.linspace(start=-torch.pi, end=torch.pi, steps=2000) # shape -> 2000 1D
y = torch.sin(input=x) # Shape -> 2000 1D

# Random Weights → for 3 degree term | shape -> Scaler
a = torch.randn((), dtype=dtype)
b = torch.randn((), dtype=dtype)
c = torch.randn((), dtype=dtype)
d = torch.randn((), dtype=dtype)

def y_pred_equation(X_train):
    y_pred = a + (b * x) + (c * x.pow(2)) + (d * x.pow(3))
    return y_pred # shape -> 2000 1D

def main():
    # Training
    for i in range(2500):
        # Prediction
        y_pred = y_pred_equation(x)

        # Loss Calculation
        loss = (y - y_pred).pow(2).sum().item()

        if i % 100 == 99:
            print(f"loss at {i + 1}it epoch → {loss}")

        # Gradients Calculation
        grad_y_pred = -2 * (y - y_pred) # shape -> 2000
        grad_a = grad_y_pred.sum()
        grad_b = torch.matmul(input = grad_y_pred, other = x).sum()
        grad_c = torch.matmul(input = grad_y_pred, other = x.pow(2)).sum()
        grad_d = torch.matmul(input = grad_y_pred, other = x.pow(3)).sum()

        # Updating
        global a, b, c, d
        a -= learning_rate * grad_a
        b -= learning_rate * grad_b
        c -= learning_rate * grad_c
        d -= learning_rate * grad_d

In [136]:
main()

loss at 100it epoch → 5049.65478515625
loss at 200it epoch → 3342.24951171875
loss at 300it epoch → 2213.177490234375
loss at 400it epoch → 1466.541015625
loss at 500it epoch → 972.8009033203125
loss at 600it epoch → 646.296630859375
loss at 700it epoch → 430.3824462890625
loss at 800it epoch → 287.599853515625
loss at 900it epoch → 193.17823791503906
loss at 1000it epoch → 130.73727416992188
loss at 1100it epoch → 89.44469451904297
loss at 1200it epoch → 62.13767623901367
loss at 1300it epoch → 44.07936477661133
loss at 1400it epoch → 32.136985778808594
loss at 1500it epoch → 24.239397048950195
loss at 1600it epoch → 19.016551971435547
loss at 1700it epoch → 15.562531471252441
loss at 1800it epoch → 13.278234481811523
loss at 1900it epoch → 11.767545700073242
loss at 2000it epoch → 10.768451690673828
loss at 2100it epoch → 10.107705116271973
loss at 2200it epoch → 9.670716285705566
loss at 2300it epoch → 9.38168716430664
loss at 2400it epoch → 9.190545082092285
loss at 2500it epoch → 